# Agent

## Ai Agent

목표를 달성하기 위하여 주변 환경은 관찰(Observation) 하고, 보유한 도구(tools) 를 활용하여 행동 (action) 하는자율적인 어플리케이션이며, 인간의 명시적인 지시 없이도 스스로 목표를 판단하고 해당 목표를 달성하기 위한 최적의행동을 계획 함

agent란 목표(task) 달성을 위해 **Tool**을 활용해서 action하는 자율적인 어플리케이션이다.
즉, 인간의 구체적인 명시 없이도 내가 가진 tool을 활용해서 최적의 행동을 기획한다. 


navie 또는 advanced RAG는 하나하나의 목적이 있었다. 그 반면 agent RAG는 외부 소스에 대한 tool, 내부소스에 대한 tool.. 이런 식으로 구현을 하고 나의 질문을 LLM agent가 판단하여 어떤 tool을 쓰고 어떤 행동을 할지 자율적으로 결정하는 것이다.




에이전트는 랭체인 기반으로도 구현할 수 있지만, 랭그래프를 통해 구현을 할 수도 있는데, 이것을 workflow라고 말하기도 한다. (이렇게 부른다고 한다)

일반 RAG: 한번 검색 -> 한번 생성하면 끝이다. 아무리 adevanced된 RAG라도 단방향이다. 즉, 한 번의 검색과 추론으로 정답을 도출한다.

반면, Agentic Workflow(Agentic RAG)는 계획 → 도구 호출 → 관찰 → 생각 → 반복 (Loop)한다. 즉, 구조 자체가 내 질문에서 답변까지의 과정으로 다시 돌아갈 수 있는 양방향이 가능하다. -> 2단계 이상의 검색과 추론이 필요함. 
> 물론, 내가 어떻게 워크플로우를 구현하느냐에 따라서 다르다. 

>> 복합질문인 경우를 생각하면 정확한 답을 위해 단계별로 생각한다. 물론, advanced RAG도 단게별 생각이 가능하지만, Agent RAG와 다른 점은 검색과 추론을 여러번 할 수 있고, 다시 검증하는 과정을 거치기 때문에 일반적인 RAG보다는 더 복잡한 문제를 해결할 수 있다. 

## 에이전트 만드는 과정을 살펴보자

- create_react_agent (구식): 이 함수는 ReAct 패턴을 구현하기 위해 LLM의 응답을 일반 문자열로 받아와서, 특정 텍스트 패턴(예: Action: [...], Action Input: [...])을 정규표현식으로 파싱하는 방식이었습니다. 파싱 오류에 취약했습니다.

- create_tool_calling_agent (현대적): 이 함수는 Gemini 2.5 Flash Lite와 같은 최신 LLM들이 기본적으로 제공하는 Tool Calling (함수 호출) 기능을 활용합니다. 이는 LLM이 도구 사용을 결정하면, 파싱 오류 걱정 없이 구조화된 JSON 객체를 출력하도록 지시합니다. 이 방식이 더 안정적이고 빠릅니다.

agent 구현을 위해 우리는 tool을 구현해야 한다. 그리고 agent와 tool이 서로 상호작용하는 구조를 만들어야 한다. 

물론, tool을 직접 커스텀할 수도 있지만, 기본적으로 내장되어 있는 빌트인 tool이 있다!
> REPLtool: 파이썬 코드를 실행하고 결과를 반환할 수 있다. 

아래의 예시부터 여러 tool을 살펴보자. 단, tool을 통해 구현한 에이전틴 워크플로우는 아님에 주의해야 한다. 즉, 구조상 고정된 체인이다.

사용자의 입력 -> 프롬프트 -> llm 동작 -> 파싱 -> ....  이렇게 항상 고정된 순서로 실행된다. LLM이 여러 도구 중 무엇을 쓸지 선택하는 구조는 아니다. 

에이전트 라면 
>   사용자 질문을 보고
  → 어떤 도구를 쓸지 판단하고
  → 도구 입력을 만들고
  → 도구 실행 결과를 보고
  → 다음 행동을 다시 결정하고
  → 최종 답변

##### Built-in tool

In [ ]:
import os
import time
from dotenv import load_dotenv


load_dotenv()

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정하세요.")

os.environ["GOOGLE_API_KEY"] = api_key

In [ ]:
# 이 도구는 Python 코드를 REPL(Read-Eval-Print Loop) 환경에서 실행하기 위한 클래스를 제공
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_experimental.tools import PythonREPLTool

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

# 파이썬 코드를 실행하는 도구를 생성
python_tool = PythonREPLTool()

# 파이썬 코드를 실행하고 결과를 반환합니다.
print(python_tool.invoke("print(100 + 200)"))
# llm을 통해 코드를 만들고 이 코드를 REPRtool 에 넣어서 답변을 만들 수 있다.

In [ ]:

# 파이썬 코드를 실행하고 중간 과정을 출력하고 도구 실행 결과를 반환하는 함수
def print_and_execute(code, debug=False):
    if debug:
        print("CODE:")
        print(code)
    return python_tool.invoke(code)


# 파이썬 코드를 작성하도록 요청하는 프롬프트
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert python programmer, well versed in meta-programming and elegant, concise and short but well documented code. You follow the PEP8 style guide. "
            "Return only the code, no intro, no explanation, no chatty, no markdown, no code block, no nothing. Just the code.",
        ),
        ("human", "{input}"),
    ]
)

# 프롬프트와 LLM 모델을 사용하여 체인 생성
chain = prompt | llm | StrOutputParser() | RunnableLambda(print_and_execute) # RunnableLambda()는 일반 파이썬 함수를 LangChain 체인에서 쓸 수 있는 Runnable 객체로 바꿔주는 도구


# 결과 출력
print(chain.invoke("무작위 숫자 10개를 뽑아줘"))

##### Custom tool 
- tool decorator

우리는 함수를 @tool 데코레이터로 감싸서 LangChain Tool 객체로 변환할 수 있다.

  LangChain에서 내가 만든 로직을 Tool로 만드는 방식은 보통 두 가지입니다.

  1. BaseTool을 상속해서 직접 Tool 클래스를 만든다
  2. 일반 파이썬 함수에 @tool 데코레이터를 붙여 Tool로 변환한다



———

  예를 들어 일반 함수가 있습니다.
```python
  def add_numbers(a: int, b: int) -> int:
      return a + b
```

  이건 그냥 파이썬 함수입니다.

  LangChain 체인이나 에이전트에서 Tool처럼 쓰려면 @tool을 붙입니다.
```python
  from langchain_core.tools import tool

  @tool
  def add_numbers(a: int, b: int) -> int:
      """두 숫자를 더합니다."""
      return a + b
```
  이제 add_numbers는 일반 함수가 아니라 LangChain Tool 객체처럼 동작합니다.
```python
  add_numbers.invoke({"a": 3, "b": 5})
```
  결과:

  8

  즉:

  @tool 데코레이터 = 일반 함수를 LangChain Tool로 변환해주는 장치

  입니다.

  ———

  반면 “상속”은 이런 경우에 씁니다.
```python
  from langchain_core.tools import BaseTool

  class MyTool(BaseTool):
      name: str = "my_tool"
      description: str = "내가 만든 도구"

      def _run(self, query: str):
          return query.upper()
```
  이건 BaseTool이라는 클래스를 상속해서 직접 도구 클래스를 만든 것입니다.

  ———

#### 결론 
  LangChain에서는 내가 원하는 로직을 Tool로 만들 수 있다.

  방법은 크게 두 가지다.

  1. BaseTool을 상속해서 직접 Tool 클래스를 만든다.
  2. 일반 파이썬 함수에 @tool 데코레이터를 붙여 Tool 객체로 변환한다.

  @tool 데코레이터로 감싸면 해당 함수는 LangChain Tool처럼 동작하고,
  .invoke() 방식으로 실행할 수 있다.

In [ ]:
from langchain_core.tools import tool


# tool 데코레이터를 사용하여 함수를 도구로 변환합니다.
@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

print(add_numbers.invoke({"a": 3, "b": 4}))
print(multiply_numbers.invoke({"a": 3, "b": 4}))

##### Tool Binding

tool 바인딩이란 내가 만든 tool을 llm과 연결시키는 것을 말한다.

tool을 바인딩 하면 모델이 호출할 때 도구정보가 함께 연동되어서 모델이 자기가 어떤 도구를 사용해야 하고 어떻게 호출하는지 알 수 있다ㅓ

tool 바인딩된 llm에게 특정 질문을 하면? -> tool을 이용해서 행동할 수 있다. 

In [ ]:
import re
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool
from langchain_core.output_parsers import JsonOutputToolsParser
from langchain_google_genai import ChatGoogleGenerativeAI


# 도구를 정의합니다.
@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)


@tool
def add_function(a: float, b: float) -> float:
    """Adds two numbers together."""
    return a + b


@tool
def naver_news_crawl(news_url: str) -> str:
    """Crawls a 네이버 (naver.com) news article and returns the body content."""
    # HTTP GET 요청 보내기
    response = requests.get(news_url)

    # 요청이 성공했는지 확인
    if response.status_code == 200:
        # BeautifulSoup을 사용하여 HTML 파싱
        soup = BeautifulSoup(response.text, "html.parser")

        # 원하는 정보 추출
        title = soup.find("h2", id="title_area").get_text()
        content = soup.find("div", id="contents").get_text()
        cleaned_title = re.sub(r"\n{2,}", "\n", title)
        cleaned_content = re.sub(r"\n{2,}", "\n", content)
    else:
        print(f"HTTP 요청 실패. 응답 코드: {response.status_code}")

    return f"{cleaned_title}\n{cleaned_content}"


# tool 데코레이션을 통해서 파이썬 함수를 tool로 정의한 이후, tools 리스트로 만든다. 
tools = [get_word_length, add_function, naver_news_crawl]

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
llm_with_tools = llm.bind_tools(tools) # llm에게 너는 이제 이런 도구를 사용할 수 있다고 알려주는 작업이다. 

# 도구 바인딩 + 도구 파서
chain = llm_with_tools | JsonOutputToolsParser(tools=tools) # 물론, 한번에 chain으로 실행 결과까지 얻고 싶으면 RuunableLambda 같은 것을 하나 더 붙여줘야 하겠지. 

# 이전에 툴 바인딩을 했더라도, JsonOutputToolsParser에 tools를 인자로 줌 -> llm이 만든 tool call 응답을 파싱하는 역할이다. 즉, 이 부분은 도구 호출 계획을 파싱하는 부분이다. 그러므로 위의 chain을 실행하면 결과는 볼 수 없고 llm이 어떤 tool을 선택했는 지를 볼 수 있다. 

사용자의 질문 받은 llm은 그 질문 해결하려고 할 때 내부지식만으로 부족하다면? 바인됭 된 tool을 기반으로 가장 적합한 도구를 선정하고 그 도구에 전달할 인자를 정해서 답변을 한다.

In [ ]:
# 이 함수는 LLM이 이미 결정한 tool call을 받아,해당 이름에 맞는 실제 Tool을 찾아 Tool call실행하고, 그 실행 결과를 ToolMessage로 만들어 반환하는 함수다.

import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool, BaseTool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from typing import List, Callable, Any, Dict

# 도구 이름을 실제 함수 객체에 매핑 (실행을 위해 필요)
TOOL_MAP: Dict[str, Callable] = {
    t.name: t for t in tools
}   # llm이 선택한 도구 이름으로 실제 도구 객체를 찾아 실행하기 위해서 정의한다ㅏ. 



def execute_tool_calls(llm_response: AIMessage) -> List[ToolMessage]: # execute_tool_calls을 통해 호출된 결과를 메세지로 반환받는 함수
    """
    LLM의 AIMessage를 받아 포함된 모든 도구 호출을 실행하고 
    결과를 ToolMessage 리스트로 반환합니다.
    """
    if not llm_response.tool_calls:
        print("❌ LLM이 도구 호출을 결정하지 않았습니다.")
        return []

    tool_messages = []
    
    print(f"--- 💡 LLM이 {len(llm_response.tool_calls)}개의 도구 호출을 결정했습니다. ---") 

    for tool_call in llm_response.tool_calls:
        tool_name = tool_call.get('name')
        tool_args = tool_call.get('args')
        tool_call_id = tool_call.get('id')
        
        if tool_name not in TOOL_MAP:
            tool_output = f"오류: 알 수 없는 도구 '{tool_name}'입니다."
        else:
            try:
                # 🟢 수정된 부분: 직접 호출(tool_function(**tool_args)) 대신 .invoke() 사용
                tool_function: BaseTool = TOOL_MAP[tool_name]  #   tool_function = TOOL_MAP[tool_name] 인데 파이썬에게 타입 힌트를 주기 위해   tool_function 변수에는 BaseTool 계열의 LangChain Tool 객체가 들어올 것이다! 를 알려준다.  
                print(f"   -> 실행 중: {tool_name}({tool_args})")
                
                # 🛠️ 핵심 수정: .invoke()를 사용하여 도구 실행
                # **tool_args는 Python 함수 호출과 동일하게 .invoke()의 입력으로 사용됨
                tool_output: str = tool_function.invoke(tool_args) 
                
            except Exception as e:
                tool_output = f"도구 실행 중 오류 발생: {e}"

        # 실행 결과를 ToolMessage 형식으로 포맷팅 (이 부분은 유지)
        tool_message = ToolMessage(
            content=tool_output, 
            tool_call_id=tool_call_id
        )
        tool_messages.append(tool_message)
        print(f"   -> 결과: {tool_output}")

    return tool_messages 
#     return tool_messages  -> • tool_messages는 실제 도구 실행 결과를 담은 ToolMessage 객체들의 리스트입니다. 이걸 왜 굳이 ToolMessage로 감싸냐면, 이 결과를 다시 LLM에게 넘기기 위해서입니다. 이 도구 실행 결과를 다시 LLM에게 전달해서 최종 답변을 만들 수 있다. 

In [ ]:
user_question = "뉴스 기사 내용을 크롤링해줘: https://n.news.naver.com/mnews/hotissue/article/092/0002347672?type=series&cid=2000065"
llm_output = llm_with_tools.invoke([HumanMessage(content=user_question)])
executed_tool_messages = execute_tool_calls(llm_output) 

if executed_tool_messages:
    # 1. 이전 사용자 질문과 LLM의 응답(도구 호출 요청), 그리고 도구 실행 결과를 모두 컨텍스트에 담습니다.
    messages_for_final_response = [
        HumanMessage(content=user_question),
        llm_output, # LLM의 도구 호출 요청
        *executed_tool_messages # 도구 실행 결과
    ]
    
    print("\n--- 🧠 2차 LLM 호출: 도구 결과를 기반으로 최종 답변 생성 ---")
    final_response = llm.invoke(messages_for_final_response)
    
    print("\n--- ✅ 최종 답변 ---")
    print(final_response.content)

else:
    # 도구 호출이 없었다면, LLM이 바로 답변했을 것입니다.
    print("\n--- ✅ LLM이 도구 없이 바로 답변했습니다. ---")
    print(llm_output.content)

### 위의 코드 단계별 분석

LLM이 필요한 도구를 고르고 → 파이썬이 실제 도구를 실행하고 → 실행 결과를 다시 LLM에게 전달해서 최종 답변을 만드는 흐름


1. 도구함수 정의
> @tool 데코레이터를 붙여서 일반 파이썬 함수를 랭체인 tool로 바꾼다. 위의 예시에서는 총 3가지의 tool을 만들었다. 

2. 도구 리스트 생성
>   tools = [get_word_length, add_function, naver_news_crawl]

3. LLm에 tool 바인딩 
>   llm_with_tools = llm.bind_tools(tools) -> 이제 LLM은 사용자의 질문을 보고 어떤 tool을 쓸 것인지 흐름을 결정할 수 있음. 

4. 도구 호출 계획 파싱용 체인 생성 
>   chain = llm_with_tools | JsonOutputToolsParser(tools=tools)  
> 즉, 이 단계에서 도구를 실제로 실행하지는 않는다.
>  llm_with_tools가 질문을 보고 어떤 도구를 쓸지 결정하고 tool call을 생성하면   JsonOutputToolsParser(tools=tools)가 tool call을 파이썬에서 다루기 쉬운 JSON 형식으로 파싱하는 체이닝이다. 즉, 도구호출 계획을 뽑고 그걸 파싱하는 체이닝 단계이다. 실제 실행은 이후에 tool.invoke(args)를 넣어주는 부분이다. 

5. 도구 이름과 실제 Tool 객체 매핑
> LLM은 도구 이름을 문자열로 반환하기 때문에, 이 이름으로 실제 도구 객체를 찾기 위한 MAP을 만든다.

6. 도구 실행 함수 정의
>   def execute_tool_calls(llm_response: AIMessage) -> List[ToolMessage]:  
>   이 함수는 LLM이 만든 tool_calls를 받아서 실제 도구를 실행합니다.
> 내부적으로 살펴보면   llm_response.tool_calls 확인  
  → tool name 꺼내기  
  → tool args 꺼내기      
  → TOOL_MAP에서 실제 도구 찾기  
  → tool.invoke(args)로 실행  
  → 결과를 ToolMessage로 감싸기  (이 부분에서 tool 실행한 결과가 나온다. 단, 자연어로 만들기 이전 상태)

7. LLM 호출: 최종 답변 생성
> ToolMessage 안의 크롤링 결과를 읽고, 사용자에게 보여줄 자연어 답변을 만든다. 

사용자 질문 → LLM이 도구 선택 → 파이썬이 도구 실행 → 실행 결과를 ToolMessage로 포장 → LLM이 최종 답변 생성!


즉, 이 구조는 단순 체인보다는 수동으로 구현한 tool calling의 흐름이다. 만약 에이전트에게 자동 루프를 맡기면 이렇게 직접 구조를 짜는 방식과는 다를 것이다. 